In [10]:
if False:
# !pip install pandas bert-score sentence-transformers rouge-score nltk
    import nltk
    nltk.download('wordnet')
    nltk.download('omw-1.4')


In [11]:
# ============================================================
# 0. Imports and helpers
# ============================================================
import json
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np

from bert_score import score as bert_score
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score


# ============================================================
# 1. Category parsing (top / mid / bottom)
# ============================================================

def split_single_label(label: str):
    """
    Split one atomic category label into (top, mid, bottom)
    following your earlier convention:
      - split by ':'
      - top  = first part
      - mid  = second part (if exists)
      - bottom:
          * if <= 2 parts: last part
          * if 3 parts: third part
          * if >3 parts: third + rest joined with ':'
    """
    if label is None:
        return ("", "", "")
    parts = [p.strip() for p in str(label).split(":")]
    if not parts:
        return ("", "", "")

    top = parts[0] if len(parts) >= 1 else ""
    mid = parts[1] if len(parts) >= 2 else ""

    if len(parts) <= 2:
        bottom = parts[-1] if parts else ""
    elif len(parts) == 3:
        bottom = parts[-1]
    else:
        bottom = ":".join(parts[2:])

    return top, mid, bottom


def parse_category_field(cat_field: str):
    """
    Category field may contain multiple labels separated by '&'.
    Returns a list of (full_label, top, mid, bottom).
    """
    if pd.isna(cat_field) or not str(cat_field).strip():
        return []

    raw = str(cat_field)
    parts = [p.strip() for p in raw.split("&")]
    result = []
    for p in parts:
        if not p:
            continue
        top, mid, bottom = split_single_label(p)
        # Use full label (as-is) for bottom-level key
        full_label = p
        result.append((full_label, top, mid, bottom))
    return result


# ============================================================
# 2. Loading GT CSV and prediction JSON, merging
# ============================================================

def load_and_merge(gt_csv_path, pred_json_path):
    """
    GT CSV columns: id, video, Action, Justification, control, Category
    Predictions JSON: [{"image_id": "...", "caption": "..."}]
      where caption is the predicted justification.
    """
    gt_csv_path = Path(gt_csv_path)
    pred_json_path = Path(pred_json_path)

    df_gt = pd.read_csv(gt_csv_path)

    with open(pred_json_path, "r", encoding="utf-8") as f:
        preds = json.load(f)

    df_pred = pd.DataFrame(preds)

    # Normalize id types for merge
    # In GT: column 'id'
    # In preds: column 'image_id'
    df_gt["id"] = df_gt["id"].astype(str)
    df_pred["image_id"] = df_pred["image_id"].astype(str)

    df_merged = df_gt.merge(
        df_pred,
        left_on="id",
        right_on="image_id",
        how="inner",
        suffixes=("", "_pred")
    )

    # Keep only rows with non-empty GT justification and prediction
    df_merged["Justification"] = df_merged["Justification"].astype(str).str.strip()
    df_merged["caption"] = df_merged["caption"].astype(str).str.strip()

    df_merged = df_merged[
        (df_merged["Justification"] != "") &
        (df_merged["caption"] != "")
    ].reset_index(drop=True)

    return df_merged


# ============================================================
# 3. Metric computation (BERTScore, SBERT cosine, ROUGE-L, METEOR)
# ============================================================

# Prepare global models / scorers (to reuse across calls)
_sbert_model = None
_rouge_scorer = None

def get_sbert_model(model_name="all-MiniLM-L6-v2"):
    global _sbert_model
    if _sbert_model is None:
        _sbert_model = SentenceTransformer(model_name)
    return _sbert_model


def get_rouge_scorer():
    global _rouge_scorer
    if _rouge_scorer is None:
        _rouge_scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    return _rouge_scorer


def compute_metrics_for_pairs(refs, hyps):
    """
    refs: list of reference justifications (GT)
    hyps: list of predicted justifications
    Returns a dict with average metrics:
      - bert_P, bert_R, bert_F1
      - sbert_cosine
      - rougeL_F1
      - meteor
    """
    assert len(refs) == len(hyps)
    n = len(refs)
    if n == 0:
        return {
            "count": 0,
            "bert_P": np.nan,
            "bert_R": np.nan,
            "bert_F1": np.nan,
            "sbert_cosine": np.nan,
            "rougeL_F1": np.nan,
            "meteor": np.nan,
        }

    # BERTScore
    P, R, F1 = bert_score(
        cands=hyps,
        refs=refs,
        lang="en",
        verbose=False
    )
    bert_P = float(P.mean().item())
    bert_R = float(R.mean().item())
    bert_F1 = float(F1.mean().item())

    # SBERT cosine similarity
    sbert = get_sbert_model()
    emb_ref = sbert.encode(refs, convert_to_tensor=True)
    emb_hyp = sbert.encode(hyps, convert_to_tensor=True)

    # cosine similarity for each pair
    # (e1 * e2).sum / (|e1||e2|)
    from torch.nn import functional as F
    cos_sims = F.cosine_similarity(emb_ref, emb_hyp)
    sbert_cosine = float(cos_sims.mean().item())

    # ROUGE-L
    scorer = get_rouge_scorer()
    rouge_scores = []
    for r, h in zip(refs, hyps):
        score = scorer.score(r, h)["rougeL"]
        rouge_scores.append(score.fmeasure)
    rougeL_F1 = float(np.mean(rouge_scores))

    # METEOR
    meteor_scores = []
    for r, h in zip(refs, hyps):
        meteor_scores.append(
            meteor_score([r.split()], h.split())
        )
    meteor_avg = float(np.mean(meteor_scores))

    return {
        "count": n,
        "bert_P": bert_P,
        "bert_R": bert_R,
        "bert_F1": bert_F1,
        "sbert_cosine": sbert_cosine,
        "rougeL_F1": rougeL_F1,
        "meteor": meteor_avg,
    }


# ============================================================
# 4. Evaluation at different levels: all / top / bottom
# ============================================================

def evaluate_all(df_merged):
    """
    Global metrics across all examples.
    """
    refs = df_merged["Justification"].tolist()
    hyps = df_merged["caption"].tolist()
    return compute_metrics_for_pairs(refs, hyps)


def build_category_indices(df_merged):
    """
    Build indices for:
      - top-level categories
      - bottom-level categories (full label as in Category)
    If a row has multiple categories (with '&'), it is included in
    each category's list.
    """
    top_to_indices = defaultdict(list)
    bottom_to_indices = defaultdict(list)

    for idx, row in df_merged.iterrows():
        cats = parse_category_field(row.get("Category", ""))
        if not cats:
            continue

        for full_label, top, mid, bottom in cats:
            if top:
                top_to_indices[top].append(idx)
            if full_label:
                bottom_to_indices[full_label].append(idx)

    return top_to_indices, bottom_to_indices


def evaluate_by_top(df_merged):
    """
    Returns dict: top_category -> metric_dict
    """
    top_to_indices, _ = build_category_indices(df_merged)
    results = {}
    for top, indices in sorted(top_to_indices.items(), key=lambda x: x[0]):
        subset = df_merged.loc[indices]
        refs = subset["Justification"].tolist()
        hyps = subset["caption"].tolist()
        res = compute_metrics_for_pairs(refs, hyps)
        results[top] = res
    return results


def evaluate_by_bottom(df_merged):
    """
    Returns dict: bottom_full_label -> metric_dict
    bottom_full_label is the full atomic label string from Category,
    e.g. "Traffic Conditions:Traffic Light:Red light"
    """
    _, bottom_to_indices = build_category_indices(df_merged)
    results = {}
    for bottom, indices in sorted(bottom_to_indices.items(), key=lambda x: x[0]):
        subset = df_merged.loc[indices]
        refs = subset["Justification"].tolist()
        hyps = subset["caption"].tolist()
        res = compute_metrics_for_pairs(refs, hyps)
        results[bottom] = res
    return results


# ============================================================
# 5. Pretty-printing and saving to txt
# ============================================================

def format_metric_block(name, metrics_dict, indent=""):
    """
    name: section name (e.g., 'GLOBAL', 'Traffic Conditions', etc.)
    metrics_dict: dict returned by compute_metrics_for_pairs
    """
    if metrics_dict["count"] == 0:
        return f"{indent}{name}: (no examples)\n"

    s = []
    s.append(f"{indent}{name} (n={metrics_dict['count']}):")
    s.append(f"{indent}  BERTScore P  : {metrics_dict['bert_P']:.4f}")
    s.append(f"{indent}  BERTScore R  : {metrics_dict['bert_R']:.4f}")
    s.append(f"{indent}  BERTScore F1 : {metrics_dict['bert_F1']:.4f}")
    s.append(f"{indent}  SBERT cosine : {metrics_dict['sbert_cosine']:.4f}")
    s.append(f"{indent}  ROUGE-L F1   : {metrics_dict['rougeL_F1']:.4f}")
    s.append(f"{indent}  METEOR       : {metrics_dict['meteor']:.4f}")
    return "\n".join(s) + "\n"


def write_results_to_txt(
    out_path,
    global_metrics,
    top_metrics=None,
    bottom_metrics=None,
    include_top=True,
    include_bottom=False,
):
    """
    Writes global + per-top + per-bottom metrics to a txt file.
    You can choose whether to include per-top / per-bottom.
    """
    out_path = Path(out_path)
    lines = []

    # Global
    lines.append("=== GLOBAL RESULTS (all categories) ===")
    lines.append(format_metric_block("ALL", global_metrics))
    lines.append("")

    # Top-level categories
    if include_top and top_metrics is not None:
        lines.append("=== PER TOP-LEVEL CATEGORY ===")
        for top, m in top_metrics.items():
            lines.append(format_metric_block(top, m))
            lines.append("")
        lines.append("")

    # Bottom-level categories
    if include_bottom and bottom_metrics is not None:
        lines.append("=== PER BOTTOM-LEVEL CATEGORY (full labels) ===")
        for bottom, m in bottom_metrics.items():
            lines.append(format_metric_block(bottom, m))
            lines.append("")
        lines.append("")

    out_path.write_text("\n".join(lines), encoding="utf-8")
    print(f"Saved results to: {out_path}")




In [ ]:
# ============================================================
# 6. Example usage cell (set parameters here and run)
# ============================================================

# Example – you just edit these paths and rerun the cell:
GT_CSV_PATH = "../annotations/bddx_test_with_justification_category.csv"   # id,video,Action,Justification,control,Category
PRED_JSON_PATH = "predicted_justifications_gpt4o.json"           # [{"image_id": "...", "caption": "..."}]
OUT_TXT_PATH = "eval_results_gpt4o.txt"

# Load & merge
df = load_and_merge(GT_CSV_PATH, PRED_JSON_PATH)
print(f"Merged dataset size: {len(df)} rows")

# Evaluate
global_res = evaluate_all(df)
top_res = evaluate_by_top(df)
bottom_res = evaluate_by_bottom(df)

# Save to txt (by default: global + per-top; set include_bottom=True if needed)
write_results_to_txt(
    OUT_TXT_PATH,
    global_metrics=global_res,
    top_metrics=top_res,
    bottom_metrics=bottom_res,
    include_top=True,
    include_bottom=False,   # set True if you also want per-bottom
)

# If you want to inspect some results in the notebook:
print("Global metrics:\n")
print(format_metric_block("ALL", global_res))

print("\nFirst few TOP categories:\n")
for i, (k, v) in enumerate(top_res.items()):
    if i >= 5:
        break
    print(format_metric_block(k, v))
    print()


Merged dataset size: 2127 rows


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

Saved results to: eval_results_gpt4o.txt
Global metrics:

ALL (n=2127):
  BERTScore P  : 0.8744
  BERTScore R  : 0.8790
  BERTScore F1 : 0.8766
  SBERT cosine : 0.5237
  ROUGE-L F1   : 0.1911
  METEOR       : 0.1256


First few TOP categories:

Ego car driving intention (n=557):
  BERTScore P  : 0.8662
  BERTScore R  : 0.8679
  BERTScore F1 : 0.8669
  SBERT cosine : 0.4243
  ROUGE-L F1   : 0.1537
  METEOR       : 0.0958


Environment (n=14):
  BERTScore P  : 0.8789
  BERTScore R  : 0.8793
  BERTScore F1 : 0.8790
  SBERT cosine : 0.5772
  ROUGE-L F1   : 0.2039
  METEOR       : 0.1438


Interaction with Other Road Users (n=564):
  BERTScore P  : 0.8790
  BERTScore R  : 0.8791
  BERTScore F1 : 0.8790
  SBERT cosine : 0.5942
  ROUGE-L F1   : 0.1961
  METEOR       : 0.1224


Road Condition (n=317):
  BERTScore P  : 0.8742
  BERTScore R  : 0.8844
  BERTScore F1 : 0.8792
  SBERT cosine : 0.5853
  ROUGE-L F1   : 0.1771
  METEOR       : 0.1268


Traffic Conditions (n=536):
  BERTScore P  : 0.88